# Modul A · Kapitel 2 · Teil 2 — Attention

**Lernziel:** Du kannst erklären, warum eine Position im Text die Positionen vor sich einbeziehen
muss und wie sie das tut. Dafür baust du die Aufmerksamkeit in zwei Schritten selbst:

1. den gleichmäßigen Durchschnitt aller bisherigen Positionen,
2. dieselbe Rechnung mit ungleichen Gewichten — Query, Key, Value.

Das ist der markierte Kasten im Decoder-Only-Modell:

```
Eingabe
   │
   ▼
Tokenizer ────────────► Word Embedding ──┐             ┏━━━━━━━━━━━┓
   │                                     ├──► Add ──►  ┃ Attention ┃ ──► …
   └──────────────────► Positional       │             ┗━━━━━━━━━━━┛
                        Encoding ────────┘
```

Alles links davon ist im Setup fertig eingebaut.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage — kurz überlegen, gerne mit der Nachbarin / dem Nachbarn |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die Funktionen
und Klassen, die du vorher schreibst.

Es ist **eine Challenge**.

---
## 0 · Setup

▶️ Führe diese zwei Zellen aus. Die zweite bringt Tokenizer, Vokabular, Einstellungen und
Embedding fertig mit — dieses Notebook läuft für sich allein.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn import functional as F

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
torch.manual_seed(1337)

print(f"PyTorch {torch.__version__}")
print("Setup fertig ✔")

In [ ]:
def lade_faust():
    """Lädt Goethes Faust I und II — einmalig von Project Gutenberg, danach aus `faust.txt`."""
    import re
    import urllib.request

    pfad = Path("faust.txt")
    if pfad.exists():
        print(f"Gelesen: {pfad}")
        return pfad.read_text(encoding="utf-8")

    print("faust.txt nicht gefunden — lade von Project Gutenberg (rund 0,5 MB) …")

    def teil(nummer, ab):
        """Holt ein Buch und schneidet Vorspann, Nachspann und Inhaltsverzeichnis weg."""
        adresse = f"https://www.gutenberg.org/cache/epub/{nummer}/pg{nummer}.txt"
        roh = urllib.request.urlopen(adresse).read().decode("utf-8").replace("\r\n", "\n")
        roh = roh[:roh.find("*** END OF THE PROJECT GUTENBERG")]
        roh = "\n".join(z[2:] if z.startswith("  ") else z for z in roh.split("\n"))
        return roh[roh.find(ab):]

    erster = teil(2229, "Zueignung\n\n\nIhr naht")
    zweiter = teil(2230, "1.  Akt--Anmutige Gegend")
    # Faust II schreibt die Sprechernamen mit Doppelpunkt, Faust I mit Punkt — wir vereinheitlichen
    zweiter = re.sub(r"(?m)^([A-ZÄÖÜ][A-ZÄÖÜ \-]*(?:\(.*?\))?):$", r"\1.", zweiter)

    text = re.sub(r"\n{3,}", "\n\n", erster.strip() + "\n\n" + zweiter.strip()) + "\n"
    Path("faust.txt").write_text(text, encoding="utf-8")
    print("Gespeichert als: faust.txt")
    return text


text = lade_faust()

# --- Tokenizer: ein Token ist ein Zeichen ---------------------------------------------
zeichen = sorted(set(text))
vokabular_groesse = len(zeichen)
zeichen_zu_zahl = {z: i for i, z in enumerate(zeichen)}
zahl_zu_zeichen = {i: z for i, z in enumerate(zeichen)}


def kodiere(s):
    """Text → Liste von Zahlen."""
    return [zeichen_zu_zahl[z] for z in s]


def dekodiere(zahlen):
    """Liste von Zahlen → Text."""
    return "".join(zahl_zu_zeichen[i] for i in zahlen)


# --- Die Einstellungen des Modells -----------------------------------------------------
KONTEXT = 96      # wie viele Tokens das Modell höchstens gleichzeitig verarbeitet
N_EMBD = 96       # Länge der Vektoren, mit denen das Modell intern arbeitet
N_HEAD = 4        # Aufmerksamkeitsköpfe je Schicht
DROPOUT = 0.1     # Anteil der Verbindungen, der beim Training zufällig abgeschaltet wird

# --- Embedding: Token-Tabelle und Positions-Tabelle, beide als nn.Embedding -------------
token_embedding = nn.Embedding(vokabular_groesse, N_EMBD)
positions_embedding = nn.Embedding(KONTEXT, N_EMBD)


@torch.no_grad()
def eingabe_vorbereiten(s):
    """Text → (1, T, N_EMBD): Token-Vektor plus Positions-Vektor, der Eingang jedes Blocks."""
    ids = torch.tensor([kodiere(s[-KONTEXT:])])
    T = ids.shape[1]
    return token_embedding(ids) + positions_embedding(torch.arange(T))


parameter = vokabular_groesse * N_EMBD + KONTEXT * N_EMBD
print("Text:       " + f"{len(text):,}".replace(",", ".")
      + f" Zeichen, {vokabular_groesse} verschiedene")
print(f"Modell:     Kontext {KONTEXT} · {N_EMBD} Dimensionen · {N_HEAD} Köpfe")
print(f"Embedding:  {vokabular_groesse} × {N_EMBD} + {KONTEXT} × {N_EMBD} = "
      + f"{parameter:,}".replace(",", ".") + " Parameter")
print(f"Probe:      {tuple(eingabe_vorbereiten('Habe nun, ach! Philosophie').shape)}"
      "   (Batch, Länge, Dimension)")

---
## 1 · Warum braucht ein Transformer Attention?

📖 Ein Satz besteht nicht nur aus einzelnen Wörtern — **Wörter beeinflussen sich gegenseitig**.

Beispiel:

> **Der Hund jagt die Katze, weil sie wegläuft.**

Um zu verstehen, worauf sich **„sie“** bezieht, muss das Modell andere Wörter im Satz
berücksichtigen.

**Attention löst genau dieses Problem:**

* Jedes Token schaut auf die anderen Tokens.
* Es bestimmt, **welche davon gerade wichtig sind**.
* Informationen wichtiger Tokens werden stärker berücksichtigt.

👉 **Attention erzeugt kontextabhängige Bedeutungen.**

---
## 2 · Schritt 1 - Tokens mitteln

📖 Angenommen, unser Satz besteht aus vier Tokens:

| Position | Token |
| -------: | ----- |
|        0 | Der   |
|        1 | Hund  |
|        2 | jagt  |
|        3 | die   |

Für jede Position kombinieren wir die Informationen aller Tokens, die **bis dahin bereits
vorkamen**.

* Position 0 kennt nur Position 0.
* Position 1 kennt Position 0 und 1.
* Position 2 kennt Position 0, 1 und 2.
* Position 3 kennt Position 0, 1, 2 und 3.

Da zunächst alle gleich wichtig sind, bilden wir einfach den Durchschnitt.

Die entsprechenden Gewichte sehen so aus:

|               | Pos 0 | Pos 1 | Pos 2 | Pos 3 |
| ------------- | ----: | ----: | ----: | ----: |
| **Ausgabe 0** |     1 |     0 |     0 |     0 |
| **Ausgabe 1** |     ½ |     ½ |     0 |     0 |
| **Ausgabe 2** |     ⅓ |     ⅓ |     ⅓ |     0 |
| **Ausgabe 3** |     ¼ |     ¼ |     ¼ |     ¼ |

👉 **Eine Zeile sagt also: Auf welche Positionen schaut dieses Token — und wie stark?**

Dazu erstellen wir zunächst nur eine Matrix, die sagt:

* `1` → diese Position darf betrachtet werden
* `0` → diese Position ist Zukunft und damit verboten

Für vier Positionen:

|       | Pos 0 | Pos 1 | Pos 2 | Pos 3 |
| ----- | ----: | ----: | ----: | ----: |
| Pos 0 |     1 |     0 |     0 |     0 |
| Pos 1 |     1 |     1 |     0 |     0 |
| Pos 2 |     1 |     1 |     1 |     0 |
| Pos 3 |     1 |     1 |     1 |     1 |

Diese Dreiecksform heißt **kausale Maske**. Mit PyTorch bekommen wir sie durch:

`torch.tril(...)`

Jetzt müssen daraus echte Gewichte werden.

Dafür machen wir zwei Dinge:

1. Verbotene Positionen bekommen `-∞`.
2. Anschließend wenden wir **Softmax** an.

Softmax macht aus Zahlen Gewichte, die zusammen **1 ergeben**.

Aus

`[0, 0, 0, -∞]`

wird dadurch:

`[⅓, ⅓, ⅓, 0]`

Warum?

Die drei erlaubten Positionen haben denselben Wert und bekommen deshalb dasselbe Gewicht. Die
verbotene Position mit `-∞` bekommt durch Softmax das Gewicht `0`.

### 🛠️ Challenge 1 — Die kausale Maske

Schreibe die Funktion `kausale_gewichte(T)`.

Sie soll für `T` Positionen eine Matrix erzeugen, in der jede Position **gleichmäßig auf sich
selbst und alle vorherigen Positionen schaut**.

Du bekommst bereits:

* `maske` → sagt, welche Positionen erlaubt sind
* `scores` → zunächst sind alle Positionen gleich wichtig

Deine Aufgabe:

1. Setze alle **verbotenen Positionen** auf `-∞`.
2. Verwandle die Scores mit **Softmax** in Gewichte.

**Hilfreiche Befehle:**

* `tensor.masked_fill(bedingung, wert)` → ersetzt alle Stellen, an denen `bedingung` wahr ist.
  Die Bedingung ist hier `maske == 0`, der Wert `float("-inf")`.
* `F.softmax(tensor, dim=-1)` → macht Gewichte daraus; `dim=-1` heißt „über die letzte Achse",
  also zeilenweise.

👉 **Merke dir dabei:**

> Eine Zeile = „Worauf schaut dieses Token?“

In [ ]:
def kausale_gewichte(T):
    """Jede Position schaut gleichmäßig auf sich selbst und alle Positionen davor."""

    # 1 = erlaubt, 0 = Zukunft / verboten
    maske = torch.tril(torch.ones(T, T))

    # Am Anfang sind alle erlaubten Positionen gleich wichtig
    scores = torch.zeros(T, T)

    # TODO 1:
    # Verbotene Positionen auf -∞ setzen
    scores = ...

    # TODO 2:
    # Scores zeilenweise in Gewichte umwandeln
    gewichte = ...

    return gewichte

In [ ]:
# ✅ Selbsttest
g = kausale_gewichte(8)

assert torch.is_tensor(g), "kausale_gewichte() gibt noch keinen Tensor zurück — beide TODOs offen?"
assert g.shape == (8, 8), f"Erwartet (8, 8), bekommen {tuple(g.shape)}"
assert torch.allclose(g.sum(dim=-1), torch.ones(8)), "Jede Zeile muss sich zu 1 addieren"
assert torch.allclose(g[0], torch.tensor([1.0] + [0.0] * 7)), "Position 0 hat nur sich selbst"
assert torch.allclose(g[3, :4], torch.full((4,), 0.25)), "Position 3 mittelt über 4 Stellen"
assert g[2, 5] == 0.0, "Die Zukunft muss ausgeblendet sein"

print("✅ Challenge 1 gelöst")
print()
print(kausale_gewichte(6).numpy().round(2))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def kausale_gewichte(T):
    """Jede Position schaut gleichmäßig auf sich selbst und alle Positionen davor."""
    maske = torch.tril(torch.ones(T, T))
    scores = torch.zeros(T, T)

    scores = scores.masked_fill(maske == 0, float("-inf"))

    gewichte = F.softmax(scores, dim=-1)

    return gewichte
```

Diese Dreiecksmatrix ist der Grund, warum man solche Modelle **autoregressiv** nennt: Jede
Position verwendet nur, was links von ihr steht.

</details>

▶️ Was die Matrix tut, sieht man am besten an vier einzelnen Zahlen statt an 96-dimensionalen
Vektoren.

In [ ]:
# ▶️ Vier Positionen, an jeder steht eine einzige Zahl
zahlen = torch.tensor([[10.0], [20.0], [30.0], [40.0]])
ergebnis = kausale_gewichte(4) @ zahlen

for t in range(4):
    bisher = zahlen[:t + 1].flatten().tolist()
    print(f"  Position {t}:  Durchschnitt von {str(bisher):<26} =  {ergebnis[t].item():5.1f}")

In [ ]:
# ▶️ Dieselbe Sache als Bild — links die Maske, rechts die Gewichte nach der Softmax
fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4.5))

for achse, bild, titel in [(links, torch.tril(torch.ones(8, 8)), "Die Maske: 1 erlaubt, 0 verboten"),
                           (rechts, kausale_gewichte(8), "Nach der Softmax: die Gewichte")]:
    achse.grid(False)
    achse.imshow(bild, cmap="Blues", vmin=0, vmax=1)
    for i in range(8):
        for j in range(8):
            wert = bild[i, j].item()
            achse.text(j, i, f"{wert:.2f}", ha="center", va="center", fontsize=8,
                       color="white" if wert > 0.5 else "#1f2937")
    achse.set_xlabel("bezieht Position … ein")
    achse.set_ylabel("Position …")
    achse.set_xticks(range(8))
    achse.set_yticks(range(8))
    achse.set_title(titel)

plt.tight_layout()
plt.show()

## 3 · Schritt 2 - Nicht alle Positionen sind gleich wichtig

📖 Im vorherigen Schritt haben wir einfach den **Durchschnitt aller bisherigen Positionen** gebildet.

Das Problem: Dabei bekommt jedes Wort das gleiche Gewicht.

In einem echten Satz sind aber manche Wörter viel wichtiger als andere:

> Ich ging zum **Fluss** und setzte mich auf die **Bank**.

Wenn das Modell verstehen möchte, was mit **Bank** gemeint ist, ist das Wort **Fluss** sehr wichtig.
Das Wort **und** hilft dagegen kaum weiter.

Wir möchten also nicht mehr sagen:

**„Alle vorherigen Wörter sind gleich wichtig.“**

Sondern:

**„Schau dir an, welche Wörter für die aktuelle Position besonders relevant sind.“**

Genau das macht **Attention**.

---

### 🔎 Wie entscheidet das Modell, welches Wort wichtig ist?

Dafür bekommt jedes Token drei verschiedene Vektoren:

| Vektor          | Einfache Bedeutung                         |
| --------------- | ------------------------------------------ |
| **Query** ($q$) | 🔎 **Wonach suche ich?**                   |
| **Key** ($k$)   | 🏷️ **Was kann ich anbieten?**             |
| **Value** ($v$) | 📦 **Welche Information gebe ich weiter?** |

Für das Wort **Bank** könnte man sich zum Beispiel vorstellen:

* Die **Query von „Bank“** sucht nach Hinweisen: *Geht es hier um Geld oder um einen Ort am Wasser?*
* Der **Key von „Fluss“** signalisiert: *Ich habe viel mit Wasser zu tun.*
* Query und Key passen deshalb gut zusammen.

Das Modell vergleicht dafür die **Query von „Bank“** mit den **Keys der vorherigen Wörter**.

$$q_{\text{Bank}} \cdot k_j$$

Je besser Query und Key zusammenpassen, desto größer wird der Wert.

Anschließend macht die **Softmax** daraus Gewichte:

* wichtiges Wort → großes Gewicht
* unwichtiges Wort → kleines Gewicht

Im Beispiel könnte also ungefähr entstehen:

* **Fluss** → 0,8
* **zum** → 0,1
* **und** → 0,1

Diese Gewichte bestimmen anschließend, **wie viel vom jeweiligen Value-Vektor übernommen wird**.

Das Ergebnis für **Bank** ist also nicht mehr der einfache Durchschnitt aller vorherigen Wörter, sondern ein **gewichteter Mix der relevanten Informationen**.

---

### 🧪 Wir bauen das gleich einmal von Hand

Damit man genau sehen kann, was passiert, verwenden wir zunächst sehr einfache Vektoren.

Jedes Token wird durch drei Zahlen beschrieben:

**Wasser · Geld · Rest**

Zum Beispiel könnte **Fluss** so aussehen:

$$[1,\ 0,\ 0]$$

also:

* viel **Wasser**
* kein **Geld**
* wenig sonstige Information

Damit können wir anschließend Schritt für Schritt beobachten, wie Query, Key, Gewichte und Value zusammenspielen.

In [ ]:
# ▶️ Drei Tokens im Kontext
# Wir setzen Query, Keys und Values von Hand,
# damit wir Attention Schritt für Schritt beobachten können.

namen = ["Fluss", "Konto", "und"]


# ---------------------------------------------------------
# 1. QUERY: Wonach sucht die aktuelle Position?
#
# Die drei Zahlen stehen für:
# [Wasser, Geld, Rest]
# ---------------------------------------------------------

frage_wasser = torch.tensor([
    [5.0, 0.0, 0.0]
])  # sucht stark nach Information über Wasser

frage_geld = torch.tensor([
    [0.0, 5.0, 0.0]
])  # sucht stark nach Information über Geld


# ---------------------------------------------------------
# 2. KEYS: Was bietet jedes Token an?
# ---------------------------------------------------------

keys = torch.tensor([
    [1.0, 0.0, 0.0],   # Fluss → bietet Information über Wasser an
    [0.0, 1.0, 0.0],   # Konto → bietet Information über Geld an
    [0.0, 0.0, 1.0]    # und   → bietet sonstige Information an
])


# ---------------------------------------------------------
# 3. VALUES: Was beinhaltet jedes Token?
#
# Die zwei Zahlen stehen für:
# [Ufer, Zinsen]
#
# Je höher die Aufmerksamkeit für ein Token,
# desto stärker wird sein Value übernommen.
# ---------------------------------------------------------

values = torch.tensor([
    [1.0, 0.0],   # Fluss → inhaltliche Informationen über den Fluss, wie z.B. Ufer 
    [0.0, 1.0],   # Konto → inhaltliche Informationen über Konto, wie z.B. Zinsen
    [0.0, 0.0]    # und   → gibt hier keine relevante Information weiter
])


print(
    f"Query {tuple(frage_wasser.shape)}   "
    f"Keys {tuple(keys.shape)}   "
    f"Values {tuple(values.shape)}"
)

`attention(q, k, v)` rechnet den attention score aus:

In [ ]:
def attention(q, k, v):
    """Gewichtete Summe der Values: Was zur Query passt, fließt ein."""
    gewichte = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5

    gewichte = F.softmax(gewichte, dim=-1)

    return gewichte @ v

In [ ]:
for beschreibung, q in [("Wasser", frage_wasser), ("Geld", frage_geld)]:
    gewichte = attention(q, keys, torch.eye(3))[0]
    ergebnis = attention(q, keys, values)[0]
    print(f"  Query sucht {beschreibung}:")
    print("     Gewichte:  " + "   ".join(f"{n} {g:.2f}" for n, g in zip(namen, gewichte)))
    print(f"     Ergebnis:  Ufer {ergebnis[0]:.2f}   Zinsen {ergebnis[1]:.2f}")

📖 Dieselben Keys, dieselben Values — nur die Query ist eine andere, und es kommt etwas anderes
heraus. Das ist der ganze Mechanismus: **Die Query bestimmt, welche früheren Positionen
einfließen.**

Zusammengeschrieben ist das die bekannteste Formel des letzten Jahrzehnts und worauf ChatGPT und andere moderne LLMs basieren:

$$\text{Attention}(Q, K, V) = \text{Softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}} +
\text{Maske}\right) V$$

---
## Geschafft

📖 Du hast die beiden Teile der Aufmerksamkeit selbst gerechnet: die **kausale Maske**, die jeder
Position den Blick nach rechts verbietet, und **Query, Key und Value**, die bestimmen, wie stark
jede frühere Position einfließt. Zusammen sind das genau die Bausteine der Formel oben.

☕ **Mach jetzt eine kurze Pause.** Steh auf, hol dir etwas zu trinken und warte, bis deine
Kolleginnen und Kollegen ebenfalls so weit sind. Wir gehen die Challenge zusammen durch, bevor es
mit dem nächsten Teil weitergeht.